# UERC26 — Satya Track 1 Evaluation on NITJ_Cropped Dataset

**Before running:** Upload your `NITJ_Cropped` folder to Google Drive and update `DRIVE_BASE` below.

Expected Drive structure:
```
MyDrive/
  NITJ_Cropped/
    uerc26_dataset.py
    uerc26_utils.py
    requirements.txt
    Cropped/
      image_list.csv
      dataset_split.csv
      pairs_train.csv
      pairs_val.csv
      pairs_test.csv
      1/L/Angle_variation/...tiff
      ...
    Submission/
      satya_track_1/
        config.ini
        solution.py
        model.py
        best_pruned_35per_tinyvit.pt
```

In [ ]:
# ── STEP 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── STEP 2: Set paths  ── UPDATE THIS if your folder is elsewhere on Drive ──
import os

DRIVE_BASE   = '/content/drive/MyDrive/NITJ_Cropped'   # <-- change if needed
SOLUTION_DIR = os.path.join(DRIVE_BASE, 'Submission', 'satya_track_1')

# Verify the key paths exist
assert os.path.isdir(DRIVE_BASE),   f'Not found: {DRIVE_BASE}'
assert os.path.isdir(SOLUTION_DIR), f'Not found: {SOLUTION_DIR}'
assert os.path.isfile(os.path.join(DRIVE_BASE, 'Cropped', 'image_list.csv')), \
    'image_list.csv missing from Cropped/ — run uerc26_dataset.py first'

print('Drive structure OK')
print(f'  NITJ_Cropped : {DRIVE_BASE}')
print(f'  Solution dir : {SOLUTION_DIR}')

In [ ]:
# ── STEP 3: Set working directory to satya_track_1/ ─────────────────────────
# This is critical — all relative paths in solution.py and config.ini
# (../../Cropped, ../../ for sys.path) are relative to this directory.
os.chdir(SOLUTION_DIR)
print(f'Working directory set to: {os.getcwd()}')

In [ ]:
# ── STEP 4: Install dependencies ─────────────────────────────────────────────
!pip install -q timm tqdm pillow-tiff
# Install from requirements.txt if it covers everything else
req_path = os.path.join(DRIVE_BASE, 'requirements.txt')
if os.path.exists(req_path):
    !pip install -q -r {req_path}
print('Dependencies installed.')

In [ ]:
# ── STEP 5: Verify GPU availability ──────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── STEP 6: Quick sanity check before full evaluation ────────────────────────
import sys
sys.path.insert(0, DRIVE_BASE)   # make uerc26_dataset and uerc26_utils importable

from uerc26_dataset import UERCPairwiseDataset

ROOT = os.path.join(DRIVE_BASE, 'Cropped')
ds = UERCPairwiseDataset(
    'test', root_dir=ROOT,
    full_image_list_csv='image_list.csv',
    data_split_csv='dataset_split.csv',
    pairs_csv='pairs_test.csv'
)
print(f'Test pairs   : {len(ds)}')
print(f'Test subjects: {ds.num_of_classes}')
(t1, t2), lbl = ds[0]
print(f'Tensor shape : {tuple(t1.shape)}  dtype={t1.dtype}')

In [ ]:
# ── STEP 7: Load model ────────────────────────────────────────────────────────
import configparser
from model import PrunedTinyViTWrapper

config = configparser.ConfigParser()
config.read('config.ini')

model_class_name = config.get('MODEL', 'model_class')
model_weights    = config.get('MODEL', 'weights')

solution_model = PrunedTinyViTWrapper()
model = solution_model.model

print(f'Model class : {model_class_name}')
print(f'Weights file: {model_weights}  (exists: {os.path.exists(model_weights)})')

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters  : {n_params:,}')

In [ ]:
# ── STEP 8: Run evaluation ────────────────────────────────────────────────────
import json
from solution import Solution

data_path  = config.get('DATA', 'data_path')   # ../../Cropped (relative, works from SOLUTION_DIR)
data_split = config.get('DATA', 'data_split')  # test
track      = config.get('SUBMISSION', 'track') # T1
name       = config.get('SUBMISSION', 'name')

batch_size = 64 if device == 'cuda' else 1    # tune based on GPU VRAM

solution = Solution(
    name,
    data_path,
    data_split,
    image_list_csv='image_list.csv',
    dataset_split_csv='dataset_split.csv',
    pairs_csv=f'pairs_{data_split}.csv',
    batch_size=batch_size,
    device=device
)
solution.load_model(model, model_weights)

print(f'Running evaluation on {device.upper()} with batch_size={batch_size} ...')
print(f'Pairs to evaluate: {len(solution.test_dataloader.dataset)}')

# Optional warmup (skip if in a hurry)
# solution.warmup(runs=3, evaluations_per_run=50)

results = solution.evaluate(runs=1)

print('\n── Results ─────────────────────────────────────')
for run_id, metrics in results.items():
    print(f'Run {run_id}:')
    for k, v in metrics.items():
        print(f'  {k:30s}: {v:.6f}')

In [ ]:
# ── STEP 9 (optional): Save results to Drive ──────────────────────────────────
import json
out_path = os.path.join(SOLUTION_DIR, 'metrics.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {out_path}')